In [ ]:
import importlib.metadata
print(importlib.metadata.version("pasqal-cloud"))

In [ ]:
import os

os.environ["REQUESTS_CA_BUNDLE"] = os.path.expanduser("~/corp-ca-bundle.pem")
os.environ["SSL_CERT_FILE"] = os.path.expanduser("~/corp-ca-bundle.pem")

import requests
print(requests.get("https://pasqal.eu.auth0.com/.well-known/openid-configuration").status_code)

In [ ]:
import qoolqit
from pasqal_cloud import PasqalCloudConnection
from qubosolver import (
    Instance,
    QuantumSolvingConfig,
    RemoteEmulator,
    Solver,
    SolverConfig,
    analysis,
    matrix,
)

connection = PasqalCloudConnection(
    username="",
    password="",
    project_id="",
)
print("Available devices:", list(connection.fetch_available_devices()))

Q = matrix.tensor(
    [[-0.2, 0.0, 1.0],
     [0.0, -1.0, 1.5],
     [1.0, 1.5, -0.1]]
)
instance = Instance(Q)

config = SolverConfig(
    solving=QuantumSolvingConfig(
        device=qoolqit.Device.from_connection(connection, "FRESNEL"),
        backend=RemoteEmulator(connection=connection, num_shots=100),
    )
)
solution = Solver(instance, config).solve()

print(analysis.to_dataframe([solution]))

samples = [
    ("".join(str(int(b)) for b in bits), float(c))
    for bits, c in zip(solution.bitstrings.tolist(), solution.counts.tolist())
]
print(samples)
print("best cost:", float(solution.costs.min()))

In [ ]:
#!/usr/bin/env python3
"""Solve a QUBO with the QuboToPasqal plugin, outside the QUARK pipeline."""

from __future__ import annotations

import os

# Corporate TLS interception: must be set before any HTTPS call.
CA_BUNDLE = os.path.expanduser("~/corp-ca-bundle.pem")
if os.path.exists(CA_BUNDLE):
    os.environ.setdefault("REQUESTS_CA_BUNDLE", CA_BUNDLE)
    os.environ.setdefault("SSL_CERT_FILE", CA_BUNDLE)

import argparse
import getpass

import numpy as np
from quark.interface_types import Qubo

import sys
sys.path.append("/Users/CBOLLE/Documents/Capgemini/QUARK-qubosolver/quark-plugin-pasqal/src/quark_plugin_pasqal")
from qubo_to_pasqal import QuboToPasqal


def build_qubo(path: str | None) -> Qubo:
    """Load a QUBO matrix from .npy/.csv, or fall back to a 3-variable example."""
    if path is None:
        matrix = np.array(
            [[-0.2, 0.0, 1.0],
             [0.0, -1.0, 1.5],
             [1.0, 1.5, -0.1]],
            dtype=np.float32,
        )
    elif path.endswith(".npy"):
        matrix = np.load(path).astype(np.float32)
    else:
        matrix = np.loadtxt(path, delimiter=",", dtype=np.float32)

    matrix = (matrix + matrix.T) / 2  # enforce the symmetry the plugin validates
    return Qubo.from_matrix(matrix)


def ensure_credentials() -> None:
    """Prompt for any Pasqal credential missing from the environment."""
    if not os.environ.get("QUBO_USERNAME"):
        os.environ["QUBO_USERNAME"] = input("Pasqal username: ")
    if not os.environ.get("QUBO_PROJECT_ID"):
        os.environ["QUBO_PROJECT_ID"] = input("Project id: ")
    if not os.environ.get("QUBO_PASSWORD"):
        os.environ["QUBO_PASSWORD"] = getpass.getpass("Password: ")



parser = argparse.ArgumentParser(description=__doc__)
parser.add_argument("--matrix", help="Path to a .npy or comma-separated .csv QUBO matrix")
parser.add_argument("--cloud", action="store_true", help="Run on Pasqal Cloud (EMU_FREE)")
parser.add_argument("--classical", action="store_true", help="Use the classical solver")
parser.add_argument("--device", default="FRESNEL", help="Device name for cloud runs")
parser.add_argument("--shots", type=int, default=100, help="Number of shots")
args = parser.parse_args()

if args.cloud and args.classical:
    parser.error("--cloud and --classical are mutually exclusive")

if args.cloud:
    ensure_credentials()

qubo = build_qubo(args.matrix)

plugin = QuboToPasqal(
    use_quantum=not args.classical,
    use_cloud=args.cloud,
    device_name=args.device,
    num_shots=args.shots,
)

result = plugin.preprocess(qubo)
if type(result).__name__ == "Failed":
    print(f"solve failed: {result.reason}")

distribution = plugin.postprocess(result)
if type(distribution).__name__ == "Failed":
    print(f"conversion failed: {distribution.reason}")

print("\n--- samples ---")
for bits, weight in plugin._to_samples(plugin._solution):
    print(f"  {bits}  {weight:g}")

print("\n--- metrics ---")
for key, value in plugin.get_metrics().items():
    print(f"  {key:>16}: {value}")


usage: ipykernel_launcher.py [-h] [--matrix MATRIX] [--cloud] [--classical]
                             [--device DEVICE] [--shots SHOTS]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/CBOLLE/Library/Jupyter/runtime/kernel-v3f4f1495b4a25c843e4e109865ab87239459a657c.json


SystemExit: 2